# Export Task2 report artifacts


In [ ]:
import os, shutil, pathlib, json, traceback
root = pathlib.Path('/kaggle/input')
print('TOP', list(root.iterdir()) if root.exists() else None)
listing = []
for p in root.rglob('*'):
    try:
        if p.is_file():
            listing.append({'path': str(p), 'size': p.stat().st_size})
    except Exception as e:
        listing.append({'path': str(p), 'error': str(e)})
pathlib.Path('/kaggle/working/input_listing.json').write_text(json.dumps(listing[:5000], indent=2))
print('n_files_seen', len(listing))
# print interesting
for item in listing:
    s = item.get('path','').replace('\\\\','/')
    if any(x in s for x in ['STATUS.md','final_results','summary.json','learning_curves','per_example','tables/','resources/','qualitative','dev_metrics']):
        print('HIT', item)
want = ['/tables/','/figures/','/logs/','/qualitative/','/resources/','/configs_used/','/raw/eval/','STATUS.md','failure.txt','code_fingerprint']
skip = ['/raw/data/','/checkpoints/','.jpg','.jpeg','trainer_state','adapter_model','connector.pt']
copied=[]
for item in listing:
    s = item.get('path','').replace('\\\\','/')
    if 'size' not in item: continue
    low = s.lower()
    if any(x in low for x in skip) and '/figures/' not in low: continue
    if not any(x in s for x in want): continue
    srcp = pathlib.Path(item['path'])
    if '/results/' in s:
        rel = s[s.index('/results/')+1:]
    else:
        # keep basename under results/_misc
        rel = 'results/_misc/' + srcp.name
    out = pathlib.Path('/kaggle/working') / rel
    out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(srcp, out)
    copied.append({'rel': rel, 'size': out.stat().st_size})
pathlib.Path('/kaggle/working/export_manifest.json').write_text(json.dumps({'n':len(copied),'files':copied}, indent=2))
print('copied', len(copied))
for c in copied:
    print(c['size'], c['rel'])
if not copied:
    raise SystemExit('No report artifacts found under /kaggle/input — kernel source mount may be empty')
